# Player Status and News Notebook

This notebook builds a **player availability dashboard** and a **historical status view** using public MLB data.

## Public data sources used

1. **MLB StatsAPI** (`statsapi.mlb.com`) for:
   - transactions (injured list, activations, options, recalls, etc.)
   - team roster snapshots
2. **PyBaseball / FanGraphs / Statcast data** for performance context (already used elsewhere in this repo).

> Note: PyBaseball does not currently provide a first-class "player news" endpoint. For status/news style events, MLB StatsAPI transactions are the most reliable free source.


In [ ]:
from __future__ import annotations

from datetime import date, timedelta
from pathlib import Path
from typing import Iterable
import json
import urllib.parse
import urllib.request

import pandas as pd


## Configuration

- `SEASON` controls the historical season window.
- `LOOKBACK_DAYS` controls the at-a-glance dashboard recency window.


In [ ]:
SEASON = date.today().year
LOOKBACK_DAYS = 14
SPORT_ID = 1  # MLB

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

BASE_URL = "https://statsapi.mlb.com/api/v1"


In [ ]:
def fetch_json(path: str, params: dict | None = None) -> dict:
    """Fetch JSON from the public MLB StatsAPI."""
    params = params or {}
    query = urllib.parse.urlencode(params)
    url = f"{BASE_URL}{path}"
    if query:
        url = f"{url}?{query}"

    with urllib.request.urlopen(url, timeout=60) as response:
        return json.loads(response.read().decode("utf-8"))


## Pull transaction feed (status/news proxy)

Transaction types include IL moves, activations, optioning, recalls, designated assignment, etc.


In [ ]:
# Includes a broad set of player-availability related transaction categories.
TRANSACTION_TYPES = [
    "D60",   # 60-day IL
    "D10",   # 10-day IL
    "D15",   # 15-day IL (pitchers, historical variants)
    "DTD",   # day-to-day injury
    "ACT",   # activated
    "REHAB", # rehab assignment
    "OPT",   # optioned
    "REC",   # recalled
    "DES",   # designated for assignment
    "TR",    # traded
    "SUS",   # suspended
    "RL",    # released
]

raw_transactions = fetch_json(
    "/transactions",
    {
        "sportId": SPORT_ID,
        "season": SEASON,
        "transactionTypes": ",".join(TRANSACTION_TYPES),
    },
)

transactions = pd.json_normalize(raw_transactions.get("transactions", []))
print(f"Pulled {len(transactions):,} transactions for season {SEASON}.")
transactions.head(10)


In [ ]:
if not transactions.empty:
    rename_map = {
        "person.id": "player_id",
        "person.fullName": "player_name",
        "toTeam.id": "to_team_id",
        "toTeam.name": "to_team_name",
        "fromTeam.id": "from_team_id",
        "fromTeam.name": "from_team_name",
        "typeCode": "type_code",
        "typeDesc": "type_desc",
        "description": "description",
        "date": "transaction_date",
        "effectiveDate": "effective_date",
    }

    transactions = transactions.rename(columns={k: v for k, v in rename_map.items() if k in transactions.columns})

    for dt_col in ["transaction_date", "effective_date"]:
        if dt_col in transactions.columns:
            transactions[dt_col] = pd.to_datetime(transactions[dt_col], errors="coerce")

transactions.head(10)


## At-a-glance dashboard view (recent status changes)


In [ ]:
recent_cutoff = pd.Timestamp(date.today() - timedelta(days=LOOKBACK_DAYS))

dashboard_cols = [
    "transaction_date",
    "player_name",
    "type_desc",
    "description",
    "to_team_name",
    "from_team_name",
]

existing_cols = [c for c in dashboard_cols if c in transactions.columns]

recent_status = (
    transactions.loc[transactions["transaction_date"] >= recent_cutoff, existing_cols]
    .sort_values("transaction_date", ascending=False)
    .reset_index(drop=True)
)

recent_status.head(50)


## Historical views

### 1) Transaction volume over time
Useful for spotting periods with elevated injury/roster churn.


In [ ]:
if not transactions.empty and "transaction_date" in transactions.columns:
    daily_volume = (
        transactions.assign(day=transactions["transaction_date"].dt.date)
        .groupby("day", as_index=False)
        .size()
        .rename(columns={"size": "transaction_count"})
    )
else:
    daily_volume = pd.DataFrame(columns=["day", "transaction_count"])

daily_volume.tail(20)


### 2) Player status history table
Useful for understanding a player's availability timeline across the season.


In [ ]:
player_history_cols = [c for c in ["transaction_date", "player_name", "type_desc", "description", "to_team_name", "from_team_name"] if c in transactions.columns]

player_status_history = transactions[player_history_cols].sort_values(
    ["player_name", "transaction_date"],
    ascending=[True, False],
)

player_status_history.head(100)


## Optional: join to performance features

You can merge this status feed with your existing batting/pitching pulls by `player_id` and date windows.
That gives you:

- short-term injury/reinstatement effects
- pre/post activation performance splits
- churn-adjusted playing-time expectations


In [ ]:
# Persist for reuse by other notebooks.
transactions_out = DATA_DIR / f"player_transactions_{SEASON}.parquet"
recent_out = DATA_DIR / f"player_status_recent_{SEASON}.parquet"

if not transactions.empty:
    transactions.to_parquet(transactions_out, index=False)

if not recent_status.empty:
    recent_status.to_parquet(recent_out, index=False)

print("Wrote:")
print(f" - {transactions_out}")
print(f" - {recent_out}")


## Notes on additional free sources

- **Retrosheet transactions**: great historical depth, less convenient for near-real-time updates.
- **MLB StatsAPI game feed**: can provide lineup/bench and roster status context at game level.
- **Fantasy platform APIs/pages** (ESPN/Yahoo/Fantrax): terms and availability vary; many are restricted.

For this repo, MLB StatsAPI + your existing PyBaseball performance datasets is the cleanest free stack.
